In [0]:
# Databricks notebook source
from datetime import datetime, timezone
import json
import requests
import time

# Define parameters via widgets
dbutils.widgets.text("catalog", "dbr_dev", "1. Catalog Name")
dbutils.widgets.text("schema", "valeriimatviiv_bronze", "2. Schema Name")
dbutils.widgets.text("volume", "market_radar_landing", "3. Landing Volume")
dbutils.widgets.text("secret_scope", "valerii-matviiv-scope", "4. Secret Scope")
dbutils.widgets.text("secret_key", "finnhub-api-key", "5. Secret Key")
dbutils.widgets.text("tickers", "AAPL,NVDA,MSFT,AMZN,TSLA,GOOGL,META,NFLX,AMD,AVGO,COST,PEP,CSCO,TMUS,QQQ", "6. Tickers")
dbutils.widgets.text("target_file_count", "1000", "7. Target File Count")

# Retrieve widget values
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
volume = dbutils.widgets.get("volume")
secret_scope = dbutils.widgets.get("secret_scope")
secret_key = dbutils.widgets.get("secret_key")
tickers = [t.strip() for t in dbutils.widgets.get("tickers").split(",")]
target_file_count = int(dbutils.widgets.get("target_file_count"))

landing_path = f"/Volumes/{catalog}/{schema}/{volume}/landing/finnhub_news"
api_key = dbutils.secrets.get(scope=secret_scope, key=secret_key)

In [0]:
# 1. Fetch General Market News (Phase 1 schema)
general_url = f"https://finnhub.io/api/v1/news?category=general&token={api_key}"
res_gen = requests.get(general_url)
general_articles = res_gen.json() if res_gen.status_code == 200 and isinstance(res_gen.json(), list) else []

# 2. Fetch NASDAQ-100 Company News across date range (Phase 2 schema)
today_str = datetime.now(timezone.utc).strftime("%Y-%m-%d")
from_date = "2026-01-01"

unique_articles = {}

# Store general market articles first
for article in general_articles:
    art_id = str(article.get("id"))
    if art_id not in unique_articles:
        article["_schema_phase"] = "general_market"
        unique_articles[art_id] = article

print(f"Collected {len(unique_articles)} general market articles.")

# Query tickers until target count is reached
for symbol in tickers:
    if len(unique_articles) >= target_file_count:
        break
        
    url = f"https://finnhub.io/api/v1/company-news?symbol={symbol}&from={from_date}&to={today_str}&token={api_key}"
    res = requests.get(url)
    
    if res.status_code == 200 and isinstance(res.json(), list):
        for article in res.json():
            art_id = str(article.get("id"))
            if art_id not in unique_articles:
                article["_schema_phase"] = "nasdaq100_company"
                article["index_tracker"] = "NASDAQ-100"
                unique_articles[art_id] = article
                
                if len(unique_articles) >= target_file_count:
                    break
    
    time.sleep(1) # Respect API rate limits

print(f"Total unique articles collected: {len(unique_articles)}")

# 3. Write each unique article as an individual landing JSON file
all_articles = list(unique_articles.values())[:target_file_count]

for idx, article in enumerate(all_articles):
    article["_landing_batch_id"] = idx
    article["_ingested_at"] = datetime.now(timezone.utc).isoformat()
    
    file_name = f"{landing_path}/news_article_{idx:04d}.json"
    with open(file_name, "w") as f:
        json.dump(article, f)

print(f"Successfully landed {len(all_articles)} unique files in {landing_path}.")

In [0]:
# 1. Read all landed JSON files using Spark
#landing_path = f"/Volumes/{catalog}/{schema}/{volume}/landing/finnhub_news"
#raw_df = spark.read.format("json").load(landing_path)

# 2. Inspect total count and merged schema
#print(f"Total files read by Spark: {raw_df.count()}")
#print("\nUnified Schema (Notice 'related' and 'index_tracker'):")
#raw_df.printSchema()

# 3. Compare general vs nasdaq100 schema phases
#display(
#    raw_df.select(
#        "_landing_batch_id", 
#        "_schema_phase", 
#        "category", 
#        "headline", 
#        "related", 
#        "index_tracker", 
#        "_ingested_at"
#    )
#)